# 09 · Heap / Priority Queue

> Un heap ti dà sempre il minimo in `O(1)` e inserisce/estrae in `O(log n)`. Segnali: "k più grandi/vicini/frequenti", "il prossimo da elaborare per priorità", "mediana in streaming".
> Idee per i singoli problemi: [Big-O › Heap](../Fundamentals/Big-O%20e%20strutture%20dati.md#heap-coda-di-priorità)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Kth Largest Element in a Stream | min-heap di dimensione `k`: la radice è il k-esimo più grande | `heappush`, `heappop` se `len > k`, `heap[0]` |
| Last Stone Weight | max-heap simulato | valori negati: `-x` |
| K Closest Points to Origin | chiave = distanza al quadrato | `heapq.nsmallest(k, pts, key=...)`, `x*x + y*y` |
| Kth Largest Element in an Array | heap di dimensione k (o quickselect) | `heapq.nlargest(k, nums)[-1]` |
| Task Scheduler | max-heap dei conteggi + coda dei task in pausa | `Counter`, `deque` di `(conteggio, tempo_disponibile)` |
| Design Twitter | merge dei feed recenti con un heap | `defaultdict(list)`, `defaultdict(set)`, timestamp crescente |
| Find Median from Data Stream | due heap: max-heap metà bassa, min-heap metà alta | bilanciare le dimensioni, `heappushpop` |

## Sintassi al volo (`import heapq`)

| Operazione | Sintassi | Costo |
|---|---|---|
| crea da lista (sul posto) | `heapq.heapify(xs)` | `O(n)` |
| inserisci | `heapq.heappush(h, x)` | `O(log n)` |
| estrai il minimo | `heapq.heappop(h)` | `O(log n)` |
| leggi il minimo | `h[0]` | `O(1)` |
| inserisci poi estrai | `heapq.heappushpop(h, x)` | `O(log n)` |
| estrai poi inserisci | `heapq.heapreplace(h, x)` | `O(log n)` |
| k più piccoli / grandi | `heapq.nsmallest(k, xs, key=...)` · `nlargest` | `O(n log k)` |

Python ha solo il **min-heap**. Per un max-heap inserisci `-x` e rinega quando leggi.

## Min-heap di base

Un heap è una lista con un ordinamento parziale: `h[0]` è il minimo, ma il resto **non** è ordinato. Non stamparlo aspettandoti una lista ordinata.

In [1]:
import heapq

prices = [1200, 450, 2599, 990]
heapq.heapify(prices)
print(prices[0], prices)  # -> 450 [450, 990, 2599, 1200]: solo il primo è garantito

heapq.heappush(prices, 300)
print(heapq.heappop(prices), heapq.heappop(prices))  # -> 300 450

450 [450, 990, 2599, 1200]
300 450


## Max-heap con valori negati

In [2]:
import heapq

stock = [5, 1, 8]
max_heap = [-x for x in stock]
heapq.heapify(max_heap)

largest = -heapq.heappop(max_heap)
second = -heapq.heappop(max_heap)
print(largest, second)  # -> 8 5
heapq.heappush(max_heap, -(largest - second))  # rimetti la differenza
print(-max_heap[0])     # -> 3

8 5
3


## Top-k con un heap di dimensione k

Per i **k più grandi** tieni un **min-heap** di dimensione `k`: quando supera `k`, togli il minimo. Alla fine `heap[0]` è il k-esimo più grande. Costo `O(n log k)` invece di `O(n log n)`.

In [3]:
import heapq

daily_revenue = [300, 1200, 450, 2599, 990, 1800]
k = 3
top: list[int] = []
for value in daily_revenue:
    heapq.heappush(top, value)
    if len(top) > k:
        heapq.heappop(top)  # butta il più piccolo dei k+1
print(sorted(top), top[0])  # -> [1200, 1800, 2599] 1200: il 3° più grande

print(heapq.nlargest(3, daily_revenue))  # -> [2599, 1800, 1200]

[1200, 1800, 2599] 1200
[2599, 1800, 1200]


## Priorità con tuple e chiavi

Le tuple si confrontano campo per campo: `(priorità, spareggio, dato)`. Per `nsmallest`/`nlargest` puoi invece passare `key=`.

In [4]:
import heapq

warehouses = [(3, 4), (1, -1), (-2, 2)]
print(heapq.nsmallest(2, warehouses, key=lambda p: p[0] ** 2 + p[1] ** 2))  # -> [(1, -1), (-2, 2)]

heap = []
for x, y in warehouses:
    heapq.heappush(heap, (x * x + y * y, x, y))  # la distanza al quadrato basta per confrontare
print(heapq.heappop(heap))  # -> (2, 1, -1)

[(1, -1), (-2, 2)]
(2, 1, -1)


⚠️ Non serve `math.sqrt` per confrontare distanze: il quadrato preserva l'ordine ed evita i float.

## Heap + coda: elementi "in pausa"

Quando un elemento estratto non può tornare subito disponibile (cooldown), mettilo in una `deque` insieme all'istante in cui potrà rientrare; a ogni passo di tempo, se il primo della coda è pronto, rimettilo nell'heap.

In [5]:
import heapq
from collections import Counter, deque

counts = Counter("AAABBC")
heap = [-c for c in counts.values()]
heapq.heapify(heap)
print(heap)  # -> [-3, -2, -1]

cooldown = deque()  # (conteggio negato rimasto, tempo in cui torna disponibile)
time, n = 0, 2
count = heapq.heappop(heap) + 1   # eseguo un task: -3 + 1 = -2 rimasti
cooldown.append((count, time + n))
print(cooldown[0])  # -> (-2, 2)

time = 2  # due unità dopo
if cooldown and cooldown[0][1] == time:  # il task è di nuovo disponibile
    heapq.heappush(heap, cooldown.popleft()[0])
print(sorted(heap))  # -> [-2, -2, -1]

[-3, -2, -1]
(-2, 2)
[-2, -2, -1]


## Contatore globale come timestamp

Per ordinare eventi "dal più recente" senza orologio, usa un contatore che cresce a ogni evento. `itertools.count()` lo fa per te. Con un max-heap (negato) sui timestamp ottieni i più recenti.

In [6]:
import heapq
from collections import defaultdict
from itertools import count

clock = count()  # 0, 1, 2, ...
posts: defaultdict[int, list[tuple[int, int]]] = defaultdict(list)
follows: defaultdict[int, set[int]] = defaultdict(set)

posts[1].append((next(clock), 101))
posts[2].append((next(clock), 201))
posts[1].append((next(clock), 102))
follows[1].add(2)

heap = []
for user in follows[1] | {1}:  # sé stesso + seguiti
    for time, post_id in posts[user][-10:]:  # bastano gli ultimi 10 di ciascuno
        heapq.heappush(heap, (-time, post_id))
print([heapq.heappop(heap)[1] for _ in range(len(heap))])  # -> [102, 201, 101]

[102, 201, 101]


## Due heap per la mediana

- `low`: max-heap (negato) con la metà più piccola;
- `high`: min-heap con la metà più grande;
- invariante: `len(low) == len(high)` oppure `len(low) == len(high) + 1`.

Il trucco per inserire senza casi: passa sempre dall'altro heap, poi ribilancia.

In [7]:
import heapq

low, high = [], []  # low negato
for x in (5, 15, 1, 3):
    heapq.heappush(low, -x)
    heapq.heappush(high, -heapq.heappop(low))  # il max di low passa in high
    if len(high) > len(low):
        heapq.heappush(low, -heapq.heappop(high))
print(sorted(-v for v in low), sorted(high))  # -> [1, 3] [5, 15]
print((-low[0] + high[0]) / 2)                # -> 4.0

[1, 3] [5, 15]
4.0


## Errori comuni

- Dimenticare di negare **sia** in ingresso **sia** in uscita con il max-heap.
- `heapq.heappop(h)` su heap vuoto: `IndexError`.
- Tuple con un oggetto non confrontabile e priorità uguali: `TypeError`. Aggiungi un indice di spareggio.
- `heapify` restituisce `None`: modifica la lista sul posto.
- Per il k-esimo più **grande** si usa un **min**-heap di dimensione k (e viceversa).